# Phase 3 — MIP Integration Extensions: Test Notebook

> ⚠️ **Requires a valid Gurobi licence.** Run on the licensed host only.

Tests for the three new `MIPSolver` methods added in Phase 3:

| Method | Purpose |
|--------|--------|
| `fix_assignments(fixed)` | Pin Layer-1 community assignments as hard constraints before `solve()` |
| `get_infeasibility_info()` | After infeasible result, identify which fixed classes participate in the IIS |
| `reset_fixed()` | Remove all fix constraints to allow the next iteration without rebuilding |

**Instance**: `data/source/test/bet-sum18.xml` (127 classes, fast to solve)

In [ ]:
import sys, os, logging
sys.path.insert(0, os.path.abspath('..'))

from src.utils.dataReader import PSTTReader
from src.MIP.solver import MIPSolver
from src.hybrid.special_constraint_decomposer import SpecialConstraintDecomposer
from gurobipy import GRB

print('Imports OK')

In [ ]:
# ── Shared config & helpers ─────────────────────────────────────────────────
INSTANCE = '../data/source/test/bet-sum18.xml'

TEST_CONFIG = {
    'train': {
        'MIP': {
            'time_limit':    120,
            'Threads':       4,
            'MIPGap':        0.01,
            'PoolSolutions': 1,
        }
    },
    'config': {
        'technique':        'MIP',
        'author':           'test',
        'institution':      'test',
        'country':          'test',
        'include_students': False,
    }
}

def make_logger(name='phase3_test'):
    log = logging.getLogger(name)
    log.setLevel(logging.INFO)
    if not log.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter('  [%(levelname)s] %(message)s'))
        log.addHandler(h)
    return log

def make_solver(reader, logger):
    """Create a fresh MIPSolver with a fully built model."""
    s = MIPSolver(reader, logger, TEST_CONFIG)
    s.build_model()
    return s

# ── Load instance once ──────────────────────────────────────────────────────
reader = PSTTReader(INSTANCE)
logger = make_logger()

decomp = SpecialConstraintDecomposer(reader)
cls_result = decomp.classify_classes()
test_cids = sorted(cls_result.free)[:5]

# Pre-build a fixed dict for the 5 test classes (first time/room option)
fixed_5 = {}
for cid in test_cids:
    cls = reader.classes[cid]
    rid = cls['room_options'][0]['id'] if cls.get('room_options') else None
    fixed_5[cid] = (0, rid)

print(f'Instance : {reader.problem_name}')
print(f'Classes  : {len(reader.classes)}')
print(f'Fixed    : {len(cls_result.fixed)}')
print(f'Community: {len(cls_result.community)}')
print(f'Free     : {len(cls_result.free)}')
print(f'Test CIDs: {test_cids}')
print(f'fixed_5  : {fixed_5}')

---
## T1 — `fix_assignments`: internal state checks

After calling `fix_assignments`, verify that:
- `_fixed_assignments` mirrors the input dict
- `_fix_constraints` is non-empty
- `_fix_constr_to_cid` maps every constraint name to a valid cid
- Expected constraint names (`fix_y_*`, `fix_u_*`) are present in the Gurobi model

In [ ]:
solver_t1 = make_solver(reader, logger)
solver_t1.fix_assignments(fixed_5)

# _fixed_assignments
assert solver_t1._fixed_assignments == fixed_5
print(f'_fixed_assignments: {solver_t1._fixed_assignments}')

# _fix_constraints non-empty
assert len(solver_t1._fix_constraints) > 0
print(f'_fix_constraints  : {len(solver_t1._fix_constraints)} constraints added')

# _fix_constr_to_cid keys all point to valid cids
for name, cid in solver_t1._fix_constr_to_cid.items():
    assert cid in fixed_5, f'Constraint {name} maps to unknown cid {cid}'
print(f'_fix_constr_to_cid: {len(solver_t1._fix_constr_to_cid)} entries, all valid')

# Named constraints exist in Gurobi model
model_names = {c.ConstrName for c in solver_t1.model.getConstrs()}
for cid, (tidx, rid) in fixed_5.items():
    if (cid, tidx) in solver_t1.y:
        assert f'fix_y_{cid}_{tidx}' in model_names
    if cid in solver_t1.u:
        assert f'fix_u_{cid}' in model_names
print('fix_y_* and fix_u_* present in Gurobi model ✓')

print('\nT1 PASSED')

### T1 — Diagnostic: list all fix constraints added

In [ ]:
import pandas as pd

rows = []
for name, cid in solver_t1._fix_constr_to_cid.items():
    # Determine type from name prefix
    if name.startswith('fix_y0_'):   ctype = 'y=0 (other time OFF)'
    elif name.startswith('fix_y_'):  ctype = 'y=1 (chosen time ON)'
    elif name.startswith('fix_x_'):  ctype = 'x=1 (chosen room ON)'
    elif name.startswith('fix_u_'):  ctype = 'u=0 (must assign)'
    else:                             ctype = 'unknown'
    rows.append({'Constraint Name': name, 'Class ID': cid, 'Type': ctype})

df_fix = pd.DataFrame(rows).sort_values(['Class ID','Type'])
print(df_fix.to_string(index=False))

---
## T2 — Solution complies with fixed assignments

Solve the model with 5 classes fixed. Verify each fixed class appears in the
solution with the exact forced `(time_option, room_id)`.

In [ ]:
assignments_list = solver_t1.solve()
assert assignments_list is not None, 'Solver returned None (infeasible) for valid fixed assignments'

assignments = assignments_list[0]

rows = []
for cid, (tidx, rid) in fixed_5.items():
    a = assignments.get(cid)
    assert a is not None, f'Class {cid} missing from solution'
    sol_topt, _, sol_rid, _ = a
    assert sol_topt is not None, f'Class {cid} unassigned despite being fixed'

    expected_topt = reader.classes[cid]['time_options'][tidx]
    time_ok  = (sol_topt == expected_topt)
    room_ok  = (rid is None) or (sol_rid == rid)

    assert time_ok,  f'Class {cid}: wrong time option in solution'
    assert room_ok,  f'Class {cid}: wrong room in solution (expected {rid}, got {sol_rid})'

    bits = sol_topt['optional_time_bits']
    rows.append({
        'CID':        cid,
        'Forced tidx': tidx,
        'Forced rid':  rid,
        'Sol days':   bits[1],
        'Sol start':  bits[2],
        'Sol length': bits[3],
        'Sol room':   sol_rid,
        'Time OK':    '✓',
        'Room OK':    '✓',
    })

pd.DataFrame(rows)

In [ ]:
print(f'T2 PASSED — all {len(fixed_5)} fixed classes assigned to forced (time, room)')

---
## T3 — `reset_fixed`: constraint removal and state reset

After `reset_fixed()`:
- All three tracking dicts are empty
- Gurobi model has exactly `n_fix_constraints` fewer constraints
- No `fix_*` names remain in the model

In [ ]:
n_fix_constrs       = len(solver_t1._fix_constraints)
n_before_reset      = len(solver_t1.model.getConstrs())

solver_t1.reset_fixed()

n_after_reset = len(solver_t1.model.getConstrs())

assert len(solver_t1._fix_constraints)   == 0
assert len(solver_t1._fix_constr_to_cid) == 0
assert len(solver_t1._fixed_assignments) == 0
print(f'Tracking dicts cleared ✓')

assert n_after_reset == n_before_reset - n_fix_constrs, (
    f'Expected {n_before_reset - n_fix_constrs} constrs, got {n_after_reset}'
)
print(f'Constraint count: {n_before_reset} → {n_after_reset} ({n_fix_constrs} removed) ✓')

remaining = [c.ConstrName for c in solver_t1.model.getConstrs() if c.ConstrName.startswith('fix_')]
assert remaining == [], f'Leftover fix_* constraints: {remaining}'
print('No fix_* constraints remain in model ✓')

print('\nT3 PASSED')

---
## T4 — reset + re-fix cycle (idempotency)

Run two complete fix → solve → reset → fix → solve cycles on a fresh solver.
The number of constraints added by `fix_assignments` must be the same in both cycles.

In [ ]:
solver_t4 = make_solver(reader, logger)

# ── Cycle 1 ──
solver_t4.fix_assignments(fixed_5)
n_fix_c1 = len(solver_t4._fix_constraints)
r1 = solver_t4.solve()
solver_t4.reset_fixed()
n_base_after_c1 = len(solver_t4.model.getConstrs())   # base count after cycle 1

# ── Cycle 2 ──
solver_t4.fix_assignments(fixed_5)
n_fix_c2 = len(solver_t4._fix_constraints)
r2 = solver_t4.solve()

print(f'Cycle 1: {n_fix_c1} fix constraints added')
print(f'Cycle 2: {n_fix_c2} fix constraints added')
assert n_fix_c1 == n_fix_c2, 'Fix constraint count differs between cycles'
print('Same number of fix constraints in both cycles ✓')

status_c1 = 'feasible' if r1 is not None else 'infeasible/timeout'
status_c2 = 'feasible' if r2 is not None else 'infeasible/timeout'
print(f'Cycle 1 result: {status_c1}')
print(f'Cycle 2 result: {status_c2}')
assert (r1 is None) == (r2 is None), 'Cycle results inconsistent'
print('Results consistent across cycles ✓')

print('\nT4 PASSED')

---
## T5 — Forced infeasibility → `get_infeasibility_info`

Find two classes connected by a `NotOverlap` or `DifferentTime` hard constraint.
Force both to overlapping time slots → model becomes infeasible.
Verify `get_infeasibility_info` returns the correct implicated class IDs.

In [ ]:
# Find a conflicting (c1, tidx1, c2, tidx2) pair
conflict_pair = None
for cons in reader.distributions['hard_constraints']:
    if cons['type'] in ('NotOverlap', 'DifferentTime', 'SameAttendees') and len(cons['classes']) >= 2:
        c1, c2 = cons['classes'][0], cons['classes'][1]
        tops1  = reader.classes[c1]['time_options']
        tops2  = reader.classes[c2]['time_options']
        t1bits = tops1[0]['optional_time_bits']
        for tidx2, t2 in enumerate(tops2):
            t2bits = t2['optional_time_bits']
            w_and = int(t1bits[0], 2) & int(t2bits[0], 2)
            d_and = int(t1bits[1], 2) & int(t2bits[1], 2)
            overlap = (t1bits[2] < t2bits[2] + t2bits[3]) and (t2bits[2] < t1bits[2] + t1bits[3])
            if w_and and d_and and overlap:
                conflict_pair = (c1, 0, c2, tidx2, cons['type'])
                break
    if conflict_pair:
        break

if conflict_pair is None:
    print('No overlapping time pair found in this instance — T5 skipped')
else:
    c1, tidx1, c2, tidx2, ctype = conflict_pair
    tb1 = reader.classes[c1]['time_options'][tidx1]['optional_time_bits']
    tb2 = reader.classes[c2]['time_options'][tidx2]['optional_time_bits']
    print(f'Constraint type : {ctype}')
    print(f'Class {c1} tidx={tidx1}: days={tb1[1]} start={tb1[2]} len={tb1[3]}')
    print(f'Class {c2} tidx={tidx2}: days={tb2[1]} start={tb2[2]} len={tb2[3]}')

In [ ]:
if conflict_pair is not None:
    c1, tidx1, c2, tidx2, _ = conflict_pair
    rid1 = reader.classes[c1]['room_options'][0]['id'] if reader.classes[c1].get('room_options') else None
    rid2 = reader.classes[c2]['room_options'][0]['id'] if reader.classes[c2].get('room_options') else None

    solver_t5 = make_solver(reader, logger)
    solver_t5.fix_assignments({c1: (tidx1, rid1), c2: (tidx2, rid2)})
    result_t5 = solver_t5.solve()

    if result_t5 is not None:
        print('Solver found a solution despite forced overlap.')
        print('The violated constraint may be soft — IIS check skipped.')
    else:
        assert solver_t5.model.Status == GRB.INFEASIBLE
        print(f'Model status: INFEASIBLE ✓')

        info = solver_t5.get_infeasibility_info()

        # Type checks
        assert isinstance(info, dict)
        assert isinstance(info['implicated_cids'], set)
        assert isinstance(info['violated_constraints'], list)
        print(f'Return type correct ✓')

        print(f'implicated_cids     : {info["implicated_cids"]}')
        print(f'violated_constraints: {len(info["violated_constraints"])} total')

        # At least one forced class should appear
        assert len(info['implicated_cids']) > 0
        assert info['implicated_cids'].issubset({c1, c2})
        print(f'Implicated ⊆ forced pair {{{c1}, {c2}}} ✓')
        print('\nT5 PASSED')

### T5 — Diagnostic: all IIS constraints

In [ ]:
if conflict_pair is not None and result_t5 is None:
    info = solver_t5.get_infeasibility_info()
    rows = []
    for name in info['violated_constraints']:
        is_fix = name in solver_t5._fix_constr_to_cid
        rows.append({
            'Constraint': name,
            'Is fix_*':   '✓' if is_fix else '',
            'Implies CID': solver_t5._fix_constr_to_cid.get(name, ''),
        })
    pd.DataFrame(rows)

---
## T6 — `get_infeasibility_info` on non-infeasible model (graceful)

In [ ]:
solver_t6 = make_solver(reader, logger)
solver_t6.fix_assignments({test_cids[0]: fixed_5[test_cids[0]]})
solver_t6.solve()  # expected feasible

info_empty = solver_t6.get_infeasibility_info()
assert isinstance(info_empty, dict)
assert info_empty['implicated_cids'] == set()
assert info_empty['violated_constraints'] == []
print(f'Model status: {solver_t6.model.Status} (non-INFEASIBLE)')
print(f'Returns empty result without crash ✓')
print('\nT6 PASSED')

---
## T7 — Non-existent `cid` in `fix_assignments` (graceful skip)

In [ ]:
solver_t7 = make_solver(reader, logger)
n_before = len(solver_t7.model.getConstrs())

solver_t7.fix_assignments({'NONEXISTENT_CID': (0, None)})
solver_t7.model.update()

n_after = len(solver_t7.model.getConstrs())
assert n_after == n_before, f'Unexpected constraints added: {n_after - n_before}'
print(f'Constraint count unchanged: {n_before} → {n_after} ✓')
print('Non-existent cid silently ignored ✓')
print('\nT7 PASSED')

---
## T8 — Double `fix_assignments` without reset (warning issued)

In [ ]:
solver_t8 = make_solver(reader, logger)

solver_t8.fix_assignments({test_cids[0]: fixed_5[test_cids[0]]})
n1 = len(solver_t8._fix_constraints)

# Second call without reset — should log a warning and still add constraints
print('(Expect a WARNING log line below)')
solver_t8.fix_assignments({test_cids[1]: fixed_5[test_cids[1]]})
n2 = len(solver_t8._fix_constraints)

assert n2 > n1, f'Second fix_assignments added no constraints ({n1} → {n2})'
print(f'Constraints after 1st fix: {n1}')
print(f'Constraints after 2nd fix: {n2} (+{n2 - n1}) ✓')
print('\nT8 PASSED')

---
## Summary

In [ ]:
import pandas as pd

summary = [
    ('T1', 'fix_assignments — internal state',           'PASSED'),
    ('T2', 'Solution complies with fixed assignments',   'PASSED'),
    ('T3', 'reset_fixed — constraints removed & dicts cleared', 'PASSED'),
    ('T4', 'reset + re-fix cycle idempotency',           'PASSED'),
    ('T5', 'Forced infeasibility → get_infeasibility_info', 'PASSED (or SKIPPED if no hard overlap found)'),
    ('T6', 'get_infeasibility_info on non-infeasible model', 'PASSED'),
    ('T7', 'Non-existent cid silently ignored',          'PASSED'),
    ('T8', 'Double fix_assignments warns + still works', 'PASSED'),
]

df_summary = pd.DataFrame(summary, columns=['Test', 'Description', 'Status'])
df_summary

---
## Appendix — Constraint count breakdown

Shows how many constraints of each fix type were added for the 5-class fixture.

In [ ]:
from collections import Counter

solver_app = make_solver(reader, logger)
solver_app.fix_assignments(fixed_5)

counter = Counter()
for name in solver_app._fix_constr_to_cid:
    if   name.startswith('fix_y0_'): counter['y=0 (other time OFF)'] += 1
    elif name.startswith('fix_y_'):  counter['y=1 (chosen time ON)'] += 1
    elif name.startswith('fix_x_'):  counter['x=1 (chosen room ON)'] += 1
    elif name.startswith('fix_u_'):  counter['u=0 (must assign)']    += 1

df_count = pd.DataFrame(counter.items(), columns=['Constraint type', 'Count'])
df_count['Per class (avg)'] = (df_count['Count'] / len(fixed_5)).round(1)
df_count

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(df_count['Constraint type'], df_count['Count'],
               color=['#4CAF50','#FF9800','#2196F3','#9C27B0'])
ax.bar_label(bars, padding=4)
ax.set_xlabel('Number of constraints added (for 5 classes)')
ax.set_title('fix_assignments: constraint breakdown')
plt.tight_layout()
plt.savefig('phase3_fix_constraint_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: phase3_fix_constraint_breakdown.png')